# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thar-26/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/thar-26/flyrank-ml-internship.git
%cd flyrank-ml-internship

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule prioritizes pages that are stale but still visible. The reasoning is that pages with meaningful search visibility may deserve review when they have not been updated for a long time.

Before encoding the rule, I check two signals that the rule depends on: days_since_last_update and impressions_90d. Staleness is linked to the refresh logic discussed in the session. I use bucket tables with counts and observed declining rates to check whether the signals behave as expected.

The rule will output one reason code, stale_but_visible, and the action label REVIEW_FOR_REFRESH.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Proxy label used only for evaluating the baseline.
# trend_direction and trend_pct are NOT used as rule inputs.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


# --------------------------------------------------
# SIGNAL 1: STALENESS
# --------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
)

print("SIGNAL 1: days_since_last_update")
display(staleness_table)

print("Verdict: MIXED")
print(
    "Reason: Declining rate increases through the 91-180 day bucket, "
    "but falls again for pages older than 180 days."
)


# --------------------------------------------------
# SIGNAL 2: VISIBILITY
# --------------------------------------------------

df["impressions_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 2000, 10000, float("inf")],
    labels=["0-100", "101-500", "501-2000", "2001-10000", "10001+"]
)

impressions_table = (
    df.groupby("impressions_bucket", observed=True)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
)

print("\nSIGNAL 2: impressions_90d")
display(impressions_table)

print("Verdict: MIXED")
print(
    "Reason: Middle-visibility buckets have higher declining rates "
    "than very low-visibility pages, but the rate falls again "
    "for the highest-impression bucket."
)

SIGNAL 1: days_since_last_update


,n,declining_rate
staleness_bucket,,
0-30,20480,0.511377
31-90,175,0.588571
91-180,9171,0.611057
181+,174,0.471264


Verdict: MIXED
Reason: Declining rate increases through the 91-180 day bucket, but falls again for pages older than 180 days.

SIGNAL 2: impressions_90d


,n,declining_rate
impressions_bucket,,
0-100,8006,0.389208
101-500,5279,0.604281
501-2000,6502,0.617964
2001-10000,6611,0.612918
10001+,3602,0.523598


Verdict: MIXED
Reason: Middle-visibility buckets have higher declining rates than very low-visibility pages, but the rate falls again for the highest-impression bucket.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Based on the signal checks, I use a deliberately simple baseline rule. A page receives a positive baseline score when it is 91-180 days since its last update and has at least 500 impressions in the trailing 90-day window. Among eligible pages, pages with more impressions receive higher scores.

The rule uses only days_since_last_update and impressions_90d. It does not use trend_direction, trend_pct, the evaluation label, product flags, or future-window information.

Eligible pages receive the reason code stale_visible_candidate and the action label REVIEW_FOR_REFRESH. Other pages receive the reason code not_prioritized and the action label NO_ACTION.

In [3]:
from pathlib import Path
import json

# Conditions chosen from the signal audit.
stale_window = (
    (df["days_since_last_update"] >= 91) &
    (df["days_since_last_update"] <= 180)
)

visible = df["impressions_90d"] >= 500

eligible = stale_window & visible


# Transparent baseline score:
# eligible pages are ranked by impressions.
df["baseline_score"] = (
    eligible.astype(int) * df["impressions_90d"]
)


# One reason-code column.
df["reason_code"] = "not_prioritized"
df.loc[eligible, "reason_code"] = "stale_visible_candidate"


# Action label.
df["action_label"] = "NO_ACTION"
df.loc[eligible, "action_label"] = "REVIEW_FOR_REFRESH"


# Build ranked queue.
queue = (
    df[
        [
            "content_id",
            "baseline_score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "is_declining_label",
        ]
    ]
    .sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


# Evaluation.
base_rate = df["is_declining_label"].mean()

precision_at_20 = (
    queue.head(20)["is_declining_label"].mean()
)

precision_at_50 = (
    queue.head(50)["is_declining_label"].mean()
)

eligible_count = eligible.sum()


# Write required CSV.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"

queue.drop(columns=["is_declining_label"]).to_csv(
    csv_path,
    index=False
)


# Write a small metrics receipt that CAN be committed.
metrics = {
    "rule": (
        "91-180 days since last update "
        "and impressions_90d >= 500; "
        "rank eligible pages by impressions_90d"
    ),
    "eligible_pages": int(eligible_count),
    "base_rate": float(base_rate),
    "precision_at_20": float(precision_at_20),
    "precision_at_50": float(precision_at_50),
}

metrics_path = output_dir / "baseline_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)


print(f"Eligible pages: {eligible_count:,}")
print(f"Base rate: {base_rate:.3f}")
print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")

print(f"\nWrote ranked queue: {csv_path}")
print(f"Wrote metrics receipt: {metrics_path}")

display(queue.head(10))


Eligible pages: 6,558
Base rate: 0.542
Precision@20: 0.450
Precision@50: 0.440

Wrote ranked queue: work/outputs/baseline_action_score.csv
Wrote metrics receipt: work/outputs/baseline_metrics.json


,content_id,baseline_score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,content_5fe46e04994d,517715,stale_visible_candidate,REVIEW_FOR_REFRESH,104,517715,4.2,0.14,1
1,content_2dba2b1f9536,443434,stale_visible_candidate,REVIEW_FOR_REFRESH,104,443434,27.9,0.21,0
2,content_2c2606c5d176,347399,stale_visible_candidate,REVIEW_FOR_REFRESH,104,347399,4.2,0.53,1
3,content_cb112fce36be,309910,stale_visible_candidate,REVIEW_FOR_REFRESH,104,309910,5.6,0.16,1
4,content_9532f197bbc8,309192,stale_visible_candidate,REVIEW_FOR_REFRESH,104,309192,2.0,0.87,1
5,content_36ff89c8214e,295097,stale_visible_candidate,REVIEW_FOR_REFRESH,104,295097,7.3,0.05,0
6,content_b28d1efd668f,286608,stale_visible_candidate,REVIEW_FOR_REFRESH,104,286608,26.2,0.06,0
7,content_813e88069237,233561,stale_visible_candidate,REVIEW_FOR_REFRESH,104,233561,26.2,0.06,1
8,content_c21024970297,211366,stale_visible_candidate,REVIEW_FOR_REFRESH,104,211366,5.1,0.41,0
9,content_c8e9d6ab9013,208678,stale_visible_candidate,REVIEW_FOR_REFRESH,104,208678,9.7,0.00,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 10 recommendations from the ranked queue. All ten received the action REVIEW_FOR_REFRESH and the reason code stale_visible_candidate. I treat these recommendations as decision support rather than automatic refresh decisions. The review below records why each page was prioritized and what additional evidence could make the recommendation wrong.

In [4]:
top10 = queue.head(10).copy()

def review_note(row):
    action = row["action_label"]
    reason = (
        f"{row['reason_code']}: {int(row['days_since_last_update'])} days "
        f"since update and {int(row['impressions_90d']):,} impressions"
    )

    # Confidence note uses current non-label signals only.
    if row["avg_position"] <= 10 and row["ctr"] < 0.20:
        confidence = (
            "Medium: strong visibility and a top-10 average position, "
            "but the low CTR suggests the page may need review."
        )
    elif row["avg_position"] <= 10:
        confidence = (
            "Medium: strong visibility and a relatively good average position, "
            "but the baseline does not know whether a refresh would improve performance."
        )
    else:
        confidence = (
            "Low: the page is visible and stale, but its weaker average position "
            "may reflect ranking difficulty rather than a refresh opportunity."
        )

    wrong_if = (
        "Wrong if the page is already performing as intended, the topic is stable, "
        "or refreshing the content would not improve the outcome."
    )

    return action, reason, confidence, wrong_if


reviews = []

for rank, (_, row) in enumerate(top10.iterrows(), start=1):
    action, reason, confidence, wrong_if = review_note(row)

    reviews.append(
        {
            "rank": rank,
            "content_id": row["content_id"],
            "action": action,
            "reason_code": row["reason_code"],
            "confidence_note": confidence,
            "what_would_make_it_wrong": wrong_if,
        }
    )


review_df = pd.DataFrame(reviews)

pd.set_option("display.max_colwidth", None)

display(review_df)

print("\nTOP-10 SKEPTICAL REVIEW\n")

for _, row in review_df.iterrows():
    print(
        f"Rank {row['rank']}: "
        f"Action={row['action']} | "
        f"Reason={row['reason_code']} | "
        f"Confidence={row['confidence_note']} | "
        f"Wrong if={row['what_would_make_it_wrong']}"
    )


,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a top-10 average position, but the low CTR suggests the page may need review.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
1,2,content_2dba2b1f9536,REVIEW_FOR_REFRESH,stale_visible_candidate,"Low: the page is visible and stale, but its weaker average position may reflect ranking difficulty rather than a refresh opportunity.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
2,3,content_2c2606c5d176,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a relatively good average position, but the baseline does not know whether a refresh would improve performance.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
3,4,content_cb112fce36be,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a top-10 average position, but the low CTR suggests the page may need review.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
4,5,content_9532f197bbc8,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a relatively good average position, but the baseline does not know whether a refresh would improve performance.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
5,6,content_36ff89c8214e,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a top-10 average position, but the low CTR suggests the page may need review.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
6,7,content_b28d1efd668f,REVIEW_FOR_REFRESH,stale_visible_candidate,"Low: the page is visible and stale, but its weaker average position may reflect ranking difficulty rather than a refresh opportunity.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
7,8,content_813e88069237,REVIEW_FOR_REFRESH,stale_visible_candidate,"Low: the page is visible and stale, but its weaker average position may reflect ranking difficulty rather than a refresh opportunity.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
8,9,content_c21024970297,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a relatively good average position, but the baseline does not know whether a refresh would improve performance.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."
9,10,content_c8e9d6ab9013,REVIEW_FOR_REFRESH,stale_visible_candidate,"Medium: strong visibility and a top-10 average position, but the low CTR suggests the page may need review.","Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome."



TOP-10 SKEPTICAL REVIEW

Rank 1: Action=REVIEW_FOR_REFRESH | Reason=stale_visible_candidate | Confidence=Medium: strong visibility and a top-10 average position, but the low CTR suggests the page may need review. | Wrong if=Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome.
Rank 2: Action=REVIEW_FOR_REFRESH | Reason=stale_visible_candidate | Confidence=Low: the page is visible and stale, but its weaker average position may reflect ranking difficulty rather than a refresh opportunity. | Wrong if=Wrong if the page is already performing as intended, the topic is stable, or refreshing the content would not improve the outcome.
Rank 3: Action=REVIEW_FOR_REFRESH | Reason=stale_visible_candidate | Confidence=Medium: strong visibility and a relatively good average position, but the baseline does not know whether a refresh would improve performance. | Wrong if=Wrong if the page is already performing as intended, th

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The top-10 review shows several weak recommendations. Ranks 2, 6, 7, and 9 are false positives under the evaluation proxy. The baseline prioritizes them because they are in the selected staleness window and have high impressions, but those conditions alone do not reliably identify declining pages.

Ranks 2 and 7 also have weaker average positions, which suggests that their performance may reflect ranking difficulty rather than a content-refresh opportunity. Other recommendations have relatively good average positions but may already be performing as intended. This shows an important limitation of the baseline: it ranks eligible pages mainly by traffic exposure and does not account for interactions between position, CTR, engagement, content characteristics, and other signals.

The baseline Precision@50 is 0.440, below the overall declining base rate of 0.542. I keep this negative result rather than changing the rule after looking at the evaluation labels. The baseline is now frozen and provides an honest reference for the later model.

Leakage check: the rule uses only days_since_last_update and impressions_90d. The score does not use trend_direction, trend_pct, is_declining_label, product flags, client_id, content_id, or future-window information. The proxy label is used only after ranking to evaluate the frozen baseline.

In [5]:
# --------------------------------------------------
# WEAK PICKS
# --------------------------------------------------

top10_check = queue.head(10).copy()

weak_picks = top10_check[
    top10_check["is_declining_label"] == 0
][
    [
        "content_id",
        "baseline_score",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "is_declining_label",
    ]
]

print("WEAK PICKS IN TOP 10")
print(f"False positives found: {len(weak_picks)}")
display(weak_picks)


# --------------------------------------------------
# BASELINE PERFORMANCE CHECK
# --------------------------------------------------

print("\nBASELINE PERFORMANCE")
print(f"Base rate: {base_rate:.3f}")
print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")

if precision_at_50 > base_rate:
    print("Result: Baseline Precision@50 is above the overall base rate.")
else:
    print(
        "Result: Baseline Precision@50 is below the overall base rate. "
        "The negative result is kept honestly and the rule is not changed "
        "after inspecting evaluation labels."
    )


# --------------------------------------------------
# LEAKAGE CHECK
# --------------------------------------------------

rule_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id",
}

leaked_inputs = rule_inputs.intersection(forbidden_inputs)

print("\nLEAKAGE CHECK")
print("Rule inputs:", sorted(rule_inputs))
print("Forbidden inputs used by rule:", sorted(leaked_inputs))

assert len(leaked_inputs) == 0

print("PASS: No label-derived columns, IDs, or forbidden inputs are used by the rule.")
print("The evaluation proxy is used only after ranking to measure baseline performance.")


WEAK PICKS IN TOP 10
False positives found: 4


,content_id,baseline_score,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
1,content_2dba2b1f9536,443434,104,443434,27.9,0.21,0
5,content_36ff89c8214e,295097,104,295097,7.3,0.05,0
6,content_b28d1efd668f,286608,104,286608,26.2,0.06,0
8,content_c21024970297,211366,104,211366,5.1,0.41,0



BASELINE PERFORMANCE
Base rate: 0.542
Precision@20: 0.450
Precision@50: 0.440
Result: Baseline Precision@50 is below the overall base rate. The negative result is kept honestly and the rule is not changed after inspecting evaluation labels.

LEAKAGE CHECK
Rule inputs: ['days_since_last_update', 'impressions_90d']
Forbidden inputs used by rule: []
PASS: No label-derived columns, IDs, or forbidden inputs are used by the rule.
The evaluation proxy is used only after ranking to measure baseline performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.